In [ ]:
import gymnasium as gym
import torch
import torch.optim as optim
import numpy as np
import random
import time

ModuleNotFoundError: No module named 'gymnasium'

In [ ]:

class DQNCnn(nn.Module):
    def __init__(self, input_channels=4, num_actions=4):
        super(DQNCnn, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU()
        )

        self.fc = nn.Sequential(
            nn.Linear(64 * 7 * 7, 512),
            nn.ReLU(),
            nn.Linear(512, num_actions)
        )

    def forward(self, x):
        x = x.permute(0, 3, 1, 2).float() / 255.0
        x = self.conv(x)
        x = x.reshape(x.size(0), -1)

        return self.fc(x)

class Memory:
    def __init__(self, capacity=10000):
        self.capacity = capacity
        self.memory = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)

    def __len__(self):
        return len(self.memory)



def train(model, memory, optimizer, batch_size=32, gamma=0.99, device="cuda"):
    if len(memory) < batch_size:
        return

    transitions = memory.sample(batch_size)
    batch = list(zip(*transitions))

    states = torch.FloatTensor(np.array(batch[0])).to(device)
    actions = torch.LongTensor(batch[1]).unsqueeze(1).to(device)
    rewards = torch.FloatTensor(batch[2]).unsqueeze(1).to(device)
    next_states = torch.FloatTensor(np.array(batch[3])).to(device)
    
    dones = torch.FloatTensor(batch[4]).unsqueeze(1).to(device)

    current_q_values = model(states).gather(1, actions)
    next_q_values = model(next_states).max(1)[0].unsqueeze(1)
    target_q_values = rewards + (gamma * next_q_values * (1 - dones))

    loss = nn.MSELoss()(current_q_values, target_q_values)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()



In [ ]:
def Action(state, epsilon, model, num_actions, device):
    if random.random() < epsilon:
        return random.randint(0, num_actions - 1)
    else: # Use the model to predict the best action
        state = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad():
            q_values = model(state)
        return q_values.argmax().item()
    

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    raw_env = gym.make("ALE/Breakout-v5", render_mode="rgb_array")

    env = GrayscaleResizeWrapper(raw_env, shape=(84, 84))
    env = FrameStackWrapper(env, num_stack=4)

    num_actions = env.action_space.n
    model = DQNCnn(input_channels=4, num_actions=num_actions).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    memory = Memory(capacity=500000)

    epsilon = 1.0
    epsilon_min = 0.1
    epsilon_decay = 0.995
    batch_size = 32
    gamma = 0.99
    episodes = 1000

    for episode in range(episodes):
        state, _ = env.reset()
        done = False
        total_reward = 0

        while not done:
            action = Action(state, epsilon, model, num_actions, device)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            memory.push(state, action, reward, next_state, done)

            train(model, memory, optimizer, batch_size=batch_size, gamma=gamma, device=device)
            
            state = next_state
            total_reward += reward

            if epsilon > epsilon_min:
                epsilon *= epsilon_decay

        print(f"Episode: {episode + 1}, Total Reward: {total_reward}, Epsilon: {epsilon:.4f}")

    torch.save(model.state_dict(), "dqn_model.pth")
    print("Model saved as dqn_model.pth")



if __name__ == "__main__":
    main()

In [ ]:
def play():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    raw_env = gym.make("ALE/Breakout-v5", render_mode="human")

    env = GrayscaleResizeWrapper(raw_env, shape=(84, 84))
    env = FrameStackWrapper(env, num_stack=4)

    num_actions = env.action_space.n
    model = DQNCnn(input_channels=4, num_actions=num_actions).to(device)
    model.load_state_dict(torch.load("dqn_model.pth", map_location=device))
    model.eval()

    state, _ = env.reset()
    done = False
    total_reward = 0

    print(f"Starting play session with model: {model}")

    while not done:
        action = Action(state, epsilon=0.0, model=model, num_actions=num_actions, device=device)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        state = next_state
        total_reward += reward

        time.sleep(0.01)  # Adjust the sleep time for smoother rendering

    print(f"Starting play session with model: {model}")
    print(f"Total Reward: {total_reward}")

if __name__ == "__main__":
    play()
